In [1]:
import pandas as pd
import os
import numpy as np
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
# definindo os caminhos
path = os.getenv('DATASETS_PATH')
data_path = os.getenv('DATA_PATH')

In [4]:
# carregando dataset de valuations (target do modelo 1)
df = pd.read_csv(path + '/player_valuations.csv')
df.head()

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057.0,BE1
1,342216,2001-07-20,100000,Unknown,1241.0,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126.0,TR1
3,6893,2003-12-15,900000,Galatasaray,984.0,GB1
4,10,2004-10-04,7000000,SV Werder Bremen,398.0,IT1


In [5]:
# vendo informações gerais do dataset de valuations
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 656301 entries, 0 to 656300
Data columns (total 6 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   player_id                            656301 non-null  int64  
 1   date                                 656301 non-null  object 
 2   market_value_in_eur                  656301 non-null  int64  
 3   current_club_name                    656301 non-null  object 
 4   current_club_id                      656300 non-null  float64
 5   player_club_domestic_competition_id  563289 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 30.0+ MB


In [6]:
# parse da data de valuation
df['date'] = pd.to_datetime(df['date'])

In [7]:
# carregando players_processed
df_players = pd.read_parquet(data_path + '/players_processed.parquet')
df_players.head()

,player_id,name,last_season,current_club_id,date_of_birth,height_in_cm,international_caps,international_goals,national_team_ranking_inv,position_rank,...,sub_pos_Left Winger,sub_pos_Left-Back,sub_pos_Right Midfield,sub_pos_Right Winger,sub_pos_Right-Back,sub_pos_Second Striker,foot_both,foot_left,foot_right,confederation_encoded
0,10,Miroslav Klose,2015,398,1978-06-09,184.0,0.0,0.0,0,4,...,False,False,False,False,False,False,False,False,True,1
1,26,Roman Weidenfeller,2017,16,1980-08-06,190.0,0.0,0.0,0,1,...,False,False,False,False,False,False,False,True,False,1
2,65,Dimitar Berbatov,2015,1091,1981-01-30,180.0,0.0,0.0,0,4,...,False,False,False,False,False,False,False,False,False,1
3,77,Lúcio,2012,506,1978-05-08,184.0,0.0,0.0,0,2,...,False,False,False,False,False,False,False,False,False,2
4,80,Tom Starke,2017,27,1981-03-18,194.0,0.0,0.0,0,1,...,False,False,False,False,False,False,False,False,True,1


In [8]:
# join de valuations com players_processed por player_id
df = df.merge(df_players.drop(columns=['name', 'current_club_id']), on='player_id', how='inner')
df.head()

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id,last_season,date_of_birth,height_in_cm,international_caps,...,sub_pos_Left Winger,sub_pos_Left-Back,sub_pos_Right Midfield,sub_pos_Right Winger,sub_pos_Right-Back,sub_pos_Second Striker,foot_both,foot_left,foot_right,confederation_encoded
0,405973,2000-01-20,150000,Unknown,3057.0,BE1,2017,1998-01-16,181.0,0.0,...,True,False,False,False,False,False,False,False,False,0
1,342216,2001-07-20,100000,Unknown,1241.0,SC1,2020,1998-02-13,181.0,0.0,...,False,True,False,False,False,False,False,True,False,1
2,3132,2003-12-09,400000,Dynamo Kyiv,126.0,TR1,2013,1980-03-10,168.0,0.0,...,False,False,False,False,False,False,False,True,False,1
3,6893,2003-12-15,900000,Galatasaray,984.0,GB1,2012,1983-11-09,188.0,0.0,...,False,False,False,False,False,False,False,False,True,1
4,10,2004-10-04,7000000,SV Werder Bremen,398.0,IT1,2015,1978-06-09,184.0,0.0,...,False,False,False,False,False,False,False,False,True,1


In [9]:
# calculando age em anos a partir da data de valuation e data de nascimento
df['age'] = (df['date'] - df['date_of_birth']).dt.days / 365.25
df['age_squared'] = df['age'] ** 2

In [10]:
# mapeando valuation date para season_year da temporada anterior
# temporada X cobre agosto do ano X-1 até julho do ano X
# valuation em jan-jul para temporada X-1, valuation em ago-dez até temporada X
df['season_year_prev'] = df['date'].apply(lambda d: d.year - 1 if d.month < 8 else d.year)

In [11]:
# carregando appearances_by_season
df_app = pd.read_parquet(data_path + '/appearances_by_season.parquet')
df_app.head()

,player_id,season_year,goals_season,assists_season,minutes_season,games_season,yellow_cards_season,red_cards_season,goals_per_90,assists_per_90,minutes_per_game
0,10,2012,16,3,2585,36,8,0,0.557060,0.104449,71.805556
1,10,2013,8,5,2220,29,2,0,0.324324,0.202703,76.551724
2,10,2014,16,9,2289,40,6,0,0.629096,0.353866,57.225000
3,10,2015,8,8,1714,31,3,0,0.420070,0.420070,55.290323
4,26,2012,0,0,4491,50,2,1,0.000000,0.000000,89.820000


In [12]:
# join com stats de aparições da temporada anterior à valuation
df = df.merge(df_app, left_on=['player_id', 'season_year_prev'], right_on=['player_id', 'season_year'], how='left')
df.head()

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id,last_season,date_of_birth,height_in_cm,international_caps,...,season_year,goals_season,assists_season,minutes_season,games_season,yellow_cards_season,red_cards_season,goals_per_90,assists_per_90,minutes_per_game
0,405973,2000-01-20,150000,Unknown,3057.0,BE1,2017,1998-01-16,181.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,342216,2001-07-20,100000,Unknown,1241.0,SC1,2020,1998-02-13,181.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3132,2003-12-09,400000,Dynamo Kyiv,126.0,TR1,2013,1980-03-10,168.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6893,2003-12-15,900000,Galatasaray,984.0,GB1,2012,1983-11-09,188.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10,2004-10-04,7000000,SV Werder Bremen,398.0,IT1,2015,1978-06-09,184.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# preenchendo nulos de aparições com 0 (jogador sem registro na temporada anterior)
app_cols = ['goals_season', 'assists_season', 'minutes_season', 'games_season',
            'yellow_cards_season', 'red_cards_season', 'goals_per_90', 'assists_per_90', 'minutes_per_game']
df[app_cols] = df[app_cols].fillna(0)

In [14]:
# carregando clubs_processed
df_clubs = pd.read_parquet(data_path + '/clubs_processed.parquet')
df_clubs.head()

,club_id,name,domestic_competition_id,squad_size,net_transfer_record,national_team_players,stadium_seats,league_tier,confederation
0,10,Arminia Bielefeld,L1,27,5900000.0,4,26515,3,europa
1,10004,Paris Football Club,FR1,31,-72300000.0,9,19904,3,europa
2,10010,Esporte Clube Bahia,BRA1,31,8140000.0,3,47364,5,amerika
3,1003,Leicester City,GB1,29,57300000.0,9,32259,1,europa
4,1005,Unione Sportiva Lecce,IT1,27,8620000.0,13,31559,3,europa


In [15]:
# join com clubs_processed por current_club_id
df = df.merge(
    df_clubs[['club_id', 'squad_size', 'net_transfer_record',
              'national_team_players', 'stadium_seats', 'league_tier']],
    left_on='current_club_id',
    right_on='club_id',
    how='left'
)
df.head()

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id,last_season,date_of_birth,height_in_cm,international_caps,...,red_cards_season,goals_per_90,assists_per_90,minutes_per_game,club_id,squad_size,net_transfer_record,national_team_players,stadium_seats,league_tier
0,405973,2000-01-20,150000,Unknown,3057.0,BE1,2017,1998-01-16,181.0,0.0,...,0.0,0.0,0.0,0.0,3057.0,31.0,400000.0,6.0,27221.0,4.0
1,342216,2001-07-20,100000,Unknown,1241.0,SC1,2020,1998-02-13,181.0,0.0,...,0.0,0.0,0.0,0.0,1241.0,33.0,15000.0,2.0,9512.0,4.0
2,3132,2003-12-09,400000,Dynamo Kyiv,126.0,TR1,2013,1980-03-10,168.0,0.0,...,0.0,0.0,0.0,0.0,126.0,25.0,-3750000.0,8.0,14879.0,4.0
3,6893,2003-12-15,900000,Galatasaray,984.0,GB1,2012,1983-11-09,188.0,0.0,...,0.0,0.0,0.0,0.0,984.0,25.0,-1350000.0,4.0,26850.0,1.0
4,10,2004-10-04,7000000,SV Werder Bremen,398.0,IT1,2015,1978-06-09,184.0,0.0,...,0.0,0.0,0.0,0.0,398.0,28.0,13550000.0,11.0,70634.0,3.0


In [16]:
# imputando nulos de club com valores neutros (clube sem dados mapeados)
df['league_tier'] = df['league_tier'].fillna(0).astype(int)
df['squad_size'] = df['squad_size'].fillna(0).astype(int)
df['net_transfer_record'] = df['net_transfer_record'].fillna(0.0)
df['national_team_players'] = df['national_team_players'].fillna(0).astype(int)
df['stadium_seats'] = df['stadium_seats'].fillna(0).astype(int)

In [17]:
# calculando club_computed_market_value: soma das valuations dos jogadores do clube
# usa apenas histórico anterior à data da valuation
df_val_raw = pd.read_csv(path + '/player_valuations.csv')
df_val_raw['date'] = pd.to_datetime(df_val_raw['date'])

# snapshot do valor total do elenco por clube e data
club_value_snapshot = (
    df_val_raw
    .dropna(subset=['current_club_id'])
    .sort_values('date')
    .groupby(['current_club_id', 'date'])['market_value_in_eur']
    .sum()
    .reset_index()
    .rename(columns={'market_value_in_eur': 'club_computed_market_value'})
)
club_value_snapshot['current_club_id'] = club_value_snapshot['current_club_id'].astype(float)

# para cada valuation, pega o snapshot mais recente do clube anterior à data
df = pd.merge_asof(
    df.sort_values('date'),
    club_value_snapshot.sort_values('date'),
    on='date',
    by='current_club_id',
    direction='backward'
)
df['club_computed_market_value'] = df['club_computed_market_value'].fillna(0.0)
df['club_computed_market_value'].describe()

count    6.556390e+05
mean     3.727235e+07
std      1.016663e+08
min      0.000000e+00
25%      1.050000e+06
50%      6.100000e+06
75%      2.515000e+07
max      1.377000e+09
Name: club_computed_market_value, dtype: float64

In [18]:
# criando feature year a partir da data de valuation
df['year'] = df['date'].dt.year

In [19]:
# criando log_market_value como target de treino
df['log_market_value'] = np.log1p(df['market_value_in_eur'])

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 655639 entries, 0 to 655638
Data columns (total 52 columns):
 #   Column                               Non-Null Count   Dtype         
---  ------                               --------------   -----         
 0   player_id                            655639 non-null  int64         
 1   date                                 655639 non-null  datetime64[ns]
 2   market_value_in_eur                  655639 non-null  int64         
 3   current_club_name                    655639 non-null  object        
 4   current_club_id                      655639 non-null  float64       
 5   player_club_domestic_competition_id  562679 non-null  object        
 6   last_season                          655639 non-null  int64         
 7   date_of_birth                        655639 non-null  datetime64[ns]
 8   height_in_cm                         655639 non-null  float64       
 9   international_caps                   655639 non-null  float64       
 

In [21]:
# colunas que não vão ser usadas
cols_to_drop = [
    'current_club_name',
    'current_club_id',
    'player_club_domestic_competition_id',
    'last_season',
    'date_of_birth',
    'yellow_cards_season',
    'red_cards_season',
    'season_year_prev',
    'season_year',
    'club_id',
]

df = df.drop(columns=cols_to_drop)
df.shape


(655639, 42)

In [22]:
# verificando nulos restantes no dataset final
df.isnull().sum().sort_values(ascending=False)

player_id                     0
goals_per_90                  0
foot_right                    0
confederation_encoded         0
age                           0
age_squared                   0
goals_season                  0
assists_season                0
minutes_season                0
games_season                  0
assists_per_90                0
date                          0
minutes_per_game              0
squad_size                    0
net_transfer_record           0
national_team_players         0
stadium_seats                 0
league_tier                   0
club_computed_market_value    0
year                          0
foot_left                     0
foot_both                     0
sub_pos_Second Striker        0
sub_pos_Right-Back            0
market_value_in_eur           0
height_in_cm                  0
international_caps            0
international_goals           0
national_team_ranking_inv     0
position_rank                 0
sub_pos_Attacking Midfield    0
sub_pos_

In [23]:
# vendo informações gerais do dataset final
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 655639 entries, 0 to 655638
Data columns (total 42 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   player_id                   655639 non-null  int64         
 1   date                        655639 non-null  datetime64[ns]
 2   market_value_in_eur         655639 non-null  int64         
 3   height_in_cm                655639 non-null  float64       
 4   international_caps          655639 non-null  float64       
 5   international_goals         655639 non-null  float64       
 6   national_team_ranking_inv   655639 non-null  int64         
 7   position_rank               655639 non-null  int64         
 8   sub_pos_Attacking Midfield  655639 non-null  bool          
 9   sub_pos_Central Midfield    655639 non-null  bool          
 10  sub_pos_Centre-Back         655639 non-null  bool          
 11  sub_pos_Centre-Forward      655639 non-

In [24]:
# distribuição do log_market_value (target do modelo 1)
df['log_market_value'].describe()

count    655639.000000
mean         13.276478
std           1.542514
min           0.000000
25%          12.206078
50%          13.122365
75%          14.220976
max          19.113828
Name: log_market_value, dtype: float64

In [25]:
# salvando model1_dataset.parquet
df.to_parquet(data_path + '/model1_dataset.parquet', index=False)